# E-commerce Delivery Intelligence

## Late-Delivery Prediction

This notebook develops machine-learning models to predict whether an order will be delivered late using information that would plausibly be available around the time the order is placed.

A time-based train/test split is used so that earlier historical orders are used to predict later orders, better reflecting how the model could operate in practice.

In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

In [2]:
master = pd.read_csv(
    "../data/processed/master_orders.csv",
    parse_dates=["order_purchase_timestamp"]
)

In [4]:
master.shape

(96470, 30)

In [5]:
master.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_late,actual_delivery_days,...,primary_product_category,primary_seller_state,number_of_categories,number_of_sellers,primary_payment_type,review_score,number_of_reviews,total_payment_value,max_payment_installments,number_of_payment_records
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,0,8.436574,...,housewares,SP,1,1,voucher,4.0,1.0,38.71,1.0,3.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,0,13.782037,...,perfumery,SP,1,1,boleto,4.0,1.0,141.46,1.0,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,0,9.394213,...,auto,SP,1,1,credit_card,5.0,1.0,179.12,3.0,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,0,13.208750,...,pet_shop,MG,1,1,credit_card,5.0,1.0,72.20,1.0,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,0,2.873877,...,stationery,SP,1,1,credit_card,5.0,1.0,28.62,1.0,1.0


In [6]:
features = [
    "order_value",
    "freight_value",
    "number_of_items",
    "number_of_categories",
    "number_of_sellers",
    "customer_state",
    "primary_seller_state",
    "primary_product_category",
    "primary_payment_type",
    "max_payment_installments",
    "purchase_month",
    "purchase_weekday",
    "promised_delivery_days"
]

target = "is_late"

model_data = master[
    ["order_purchase_timestamp"] + features + [target]
].copy()

In [7]:
model_data.head()

,order_purchase_timestamp,order_value,freight_value,number_of_items,number_of_categories,number_of_sellers,customer_state,primary_seller_state,primary_product_category,primary_payment_type,max_payment_installments,purchase_month,purchase_weekday,promised_delivery_days,is_late
0,2017-10-02 10:56:33,29.99,8.72,1,1,1,SP,SP,housewares,voucher,1.0,10,Monday,15.544063,0
1,2018-07-24 20:41:37,118.70,22.76,1,1,1,BA,SP,perfumery,boleto,1.0,7,Tuesday,19.137766,0
2,2018-08-08 08:38:49,159.90,19.22,1,1,1,GO,SP,auto,credit_card,3.0,8,Wednesday,26.639711,0
3,2017-11-18 19:28:06,45.00,27.20,1,1,1,RN,MG,pet_shop,credit_card,1.0,11,Saturday,26.188819,0
4,2018-02-13 21:18:39,19.90,8.72,1,1,1,SP,SP,stationery,credit_card,1.0,2,Tuesday,12.112049,0


In [8]:
model_data = model_data.sort_values(
    "order_purchase_timestamp"
).reset_index(drop=True)

In [9]:
model_data[
    "order_purchase_timestamp"
].agg(["min", "max"])

min   2016-09-15 12:16:38
max   2018-08-29 15:00:37
Name: order_purchase_timestamp, dtype: datetime64[us]

In [10]:
split_index = int(len(model_data) * 0.80)

train_data = model_data.iloc[:split_index].copy()
test_data = model_data.iloc[split_index:].copy()

In [11]:
print("Training rows:", len(train_data))
print("Testing rows:", len(test_data))

print(
    "Training period:",
    train_data["order_purchase_timestamp"].min(),
    "to",
    train_data["order_purchase_timestamp"].max()
)

print(
    "Testing period:",
    test_data["order_purchase_timestamp"].min(),
    "to",
    test_data["order_purchase_timestamp"].max()
)

Training rows: 77176
Testing rows: 19294
Training period: 2016-09-15 12:16:38 to 2018-05-26 18:16:57
Testing period: 2018-05-26 18:18:03 to 2018-08-29 15:00:37


In [12]:
X_train = train_data[features]
y_train = train_data[target]

X_test = test_data[features]
y_test = test_data[target]

In [13]:
print(
    f"Training late rate: {y_train.mean() * 100:.2f}%"
)

print(
    f"Testing late rate: {y_test.mean() * 100:.2f}%"
)

Training late rate: 8.82%
Testing late rate: 5.29%


In [14]:
numeric_features = [
    "order_value",
    "freight_value",
    "number_of_items",
    "number_of_categories",
    "number_of_sellers",
    "max_payment_installments",
    "purchase_month",
    "promised_delivery_days"
]

categorical_features = [
    "customer_state",
    "primary_seller_state",
    "primary_product_category",
    "primary_payment_type",
    "purchase_weekday"
]

In [15]:
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [16]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

In [17]:
logistic_model.fit(
    X_train,
    y_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](13,)","['order_value','freight_value','number_of_items',...,'purchase_month', 'purchase_weekday','promised_delivery_days']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,13
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``

In [18]:
logistic_predictions = logistic_model.predict(
    X_test
)

logistic_probabilities = (
    logistic_model.predict_proba(X_test)[:, 1]
)

In [19]:
print("LOGISTIC REGRESSION")
print("=" * 50)

print(
    f"Accuracy: "
    f"{accuracy_score(y_test, logistic_predictions):.3f}"
)

print(
    f"Precision: "
    f"{precision_score(y_test, logistic_predictions):.3f}"
)

print(
    f"Recall: "
    f"{recall_score(y_test, logistic_predictions):.3f}"
)

print(
    f"F1 Score: "
    f"{f1_score(y_test, logistic_predictions):.3f}"
)

print(
    f"ROC-AUC: "
    f"{roc_auc_score(y_test, logistic_probabilities):.3f}"
)

LOGISTIC REGRESSION
Accuracy: 0.555
Precision: 0.090
Recall: 0.811
F1 Score: 0.162
ROC-AUC: 0.717


In [20]:
print("CONFUSION MATRIX")
print("=" * 50)

cm = confusion_matrix(
    y_test,
    logistic_predictions
)

print(cm)

CONFUSION MATRIX
[[9883 8390]
 [ 193  828]]


In [21]:
tn, fp, fn, tp = cm.ravel()

print(f"True Negatives:  {tn:,}")
print(f"False Positives: {fp:,}")
print(f"False Negatives: {fn:,}")
print(f"True Positives:  {tp:,}")

True Negatives:  9,883
False Positives: 8,390
False Negatives: 193
True Positives:  828


In [22]:
from sklearn.metrics import average_precision_score

In [23]:
pr_auc = average_precision_score(
    y_test,
    logistic_probabilities
)

print(f"PR-AUC: {pr_auc:.3f}")

PR-AUC: 0.115


In [24]:
validation_split = int(
    len(train_data) * 0.80
)

train_inner = train_data.iloc[
    :validation_split
].copy()

validation_data = train_data.iloc[
    validation_split:
].copy()

In [25]:
print("Inner training rows:", len(train_inner))
print("Validation rows:", len(validation_data))
print("Final test rows:", len(test_data))

print()
print(
    "Inner training period:",
    train_inner["order_purchase_timestamp"].min(),
    "to",
    train_inner["order_purchase_timestamp"].max()
)

print(
    "Validation period:",
    validation_data["order_purchase_timestamp"].min(),
    "to",
    validation_data["order_purchase_timestamp"].max()
)

print(
    "Test period:",
    test_data["order_purchase_timestamp"].min(),
    "to",
    test_data["order_purchase_timestamp"].max()
)

Inner training rows: 61740
Validation rows: 15436
Final test rows: 19294

Inner training period: 2016-09-15 12:16:38 to 2018-03-20 09:15:24
Validation period: 2018-03-20 09:16:16 to 2018-05-26 18:16:57
Test period: 2018-05-26 18:18:03 to 2018-08-29 15:00:37


In [26]:
X_train_inner = train_inner[features]
y_train_inner = train_inner[target]

X_validation = validation_data[features]
y_validation = validation_data[target]

In [27]:
logistic_validation_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

In [28]:
logistic_validation_model.fit(
    X_train_inner,
    y_train_inner
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](13,)","['order_value','freight_value','number_of_items',...,'purchase_month', 'purchase_weekday','promised_delivery_days']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,13
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``

In [29]:
validation_probabilities = (
    logistic_validation_model
    .predict_proba(X_validation)[:, 1]
)

In [30]:
threshold_results = []

for threshold in [
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80
]:
    
    predictions = (
        validation_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_validation,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_validation,
            predictions,
            zero_division=0
        ),
        "F1": f1_score(
            y_validation,
            predictions,
            zero_division=0
        )
    })

threshold_results = pd.DataFrame(
    threshold_results
)

threshold_results

,Threshold,Precision,Recall,F1
0,0.30,0.088706,0.980016,0.162686
1,0.35,0.095017,0.966427,0.173023
2,0.40,0.102637,0.917666,0.184625
3,0.45,0.113833,0.843325,0.200589
4,0.50,0.130326,0.748201,0.221985
5,0.55,0.155308,0.646683,0.250464
6,0.60,0.182787,0.534772,0.272450
7,0.65,0.206827,0.411671,0.275327
8,0.70,0.231847,0.290967,0.258065
9,0.75,0.263889,0.182254,0.215603


In [31]:
best_threshold_row = (
    threshold_results
    .sort_values(
        "F1",
        ascending=False
    )
    .iloc[0]
)

best_threshold_row

Threshold    0.650000
Precision    0.206827
Recall       0.411671
F1           0.275327
Name: 7, dtype: float64

In [32]:
best_threshold = (
    best_threshold_row["Threshold"]
)

print(
    f"Best validation threshold: "
    f"{best_threshold:.2f}"
)

Best validation threshold: 0.65


In [33]:
import plotly.express as px

threshold_long = (
    threshold_results
    .melt(
        id_vars="Threshold",
        value_vars=[
            "Precision",
            "Recall",
            "F1"
        ],
        var_name="Metric",
        value_name="Score"
    )
)

fig = px.line(
    threshold_long,
    x="Threshold",
    y="Score",
    color="Metric",
    markers=True,
    title="Classification Threshold Trade-off"
)

fig.show()

In [34]:
random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=12,
                min_samples_leaf=5,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [35]:
random_forest_model.fit(
    X_train_inner,
    y_train_inner
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](13,)","['order_value','freight_value','number_of_items',...,'purchase_month', 'purchase_weekday','promised_delivery_days']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,13
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``

In [36]:
rf_validation_probabilities = (
    random_forest_model
    .predict_proba(X_validation)[:, 1]
)

rf_validation_predictions = (
    rf_validation_probabilities >= 0.5
).astype(int)

In [37]:
print("RANDOM FOREST - VALIDATION")
print("=" * 50)

print(
    f"Accuracy: "
    f"{accuracy_score(
        y_validation,
        rf_validation_predictions
    ):.3f}"
)

print(
    f"Precision: "
    f"{precision_score(
        y_validation,
        rf_validation_predictions,
        zero_division=0
    ):.3f}"
)

print(
    f"Recall: "
    f"{recall_score(
        y_validation,
        rf_validation_predictions,
        zero_division=0
    ):.3f}"
)

print(
    f"F1 Score: "
    f"{f1_score(
        y_validation,
        rf_validation_predictions,
        zero_division=0
    ):.3f}"
)

print(
    f"ROC-AUC: "
    f"{roc_auc_score(
        y_validation,
        rf_validation_probabilities
    ):.3f}"
)

print(
    f"PR-AUC: "
    f"{average_precision_score(
        y_validation,
        rf_validation_probabilities
    ):.3f}"
)

RANDOM FOREST - VALIDATION
Accuracy: 0.779
Precision: 0.175
Recall: 0.463
F1 Score: 0.254
ROC-AUC: 0.690
PR-AUC: 0.184


In [39]:
# RANDOM FOREST THRESHOLD TUNING

rf_threshold_results = []

for threshold in [
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80
]:

    predictions = (
        rf_validation_probabilities >= threshold
    ).astype(int)

    rf_threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_validation,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_validation,
            predictions,
            zero_division=0
        ),
        "F1": f1_score(
            y_validation,
            predictions,
            zero_division=0
        )
    })

rf_threshold_results = pd.DataFrame(
    rf_threshold_results
)

best_rf_threshold_row = (
    rf_threshold_results
    .sort_values("F1", ascending=False)
    .iloc[0]
)

print("=" * 60)
print("LOGISTIC REGRESSION - BEST VALIDATION RESULT")
print("=" * 60)

print(f"Threshold: {best_threshold_row['Threshold']:.2f}")
print(f"Precision: {best_threshold_row['Precision']:.3f}")
print(f"Recall:    {best_threshold_row['Recall']:.3f}")
print(f"F1 Score:  {best_threshold_row['F1']:.3f}")


print("\n" + "=" * 60)
print("RANDOM FOREST - BEST VALIDATION RESULT")
print("=" * 60)

print(f"Threshold: {best_rf_threshold_row['Threshold']:.2f}")
print(f"Precision: {best_rf_threshold_row['Precision']:.3f}")
print(f"Recall:    {best_rf_threshold_row['Recall']:.3f}")
print(f"F1 Score:  {best_rf_threshold_row['F1']:.3f}")


print("\n" + "=" * 60)
print("MODEL RANKING METRICS")
print("=" * 60)

logistic_validation_roc_auc = roc_auc_score(
    y_validation,
    validation_probabilities
)

logistic_validation_pr_auc = average_precision_score(
    y_validation,
    validation_probabilities
)

rf_validation_roc_auc = roc_auc_score(
    y_validation,
    rf_validation_probabilities
)

rf_validation_pr_auc = average_precision_score(
    y_validation,
    rf_validation_probabilities
)

print("Logistic Regression")
print(f"ROC-AUC: {logistic_validation_roc_auc:.3f}")
print(f"PR-AUC:  {logistic_validation_pr_auc:.3f}")

print()

print("Random Forest")
print(f"ROC-AUC: {rf_validation_roc_auc:.3f}")
print(f"PR-AUC:  {rf_validation_pr_auc:.3f}")

LOGISTIC REGRESSION - BEST VALIDATION RESULT
Threshold: 0.65
Precision: 0.207
Recall:    0.412
F1 Score:  0.275

RANDOM FOREST - BEST VALIDATION RESULT
Threshold: 0.50
Precision: 0.175
Recall:    0.463
F1 Score:  0.254

MODEL RANKING METRICS
Logistic Regression
ROC-AUC: 0.724
PR-AUC:  0.186

Random Forest
ROC-AUC: 0.690
PR-AUC:  0.184


## Final Model Selection

Logistic Regression was selected as the final model because it outperformed Random Forest on the validation period across ROC-AUC, PR-AUC and F1 score.

A classification threshold of 0.65 was selected using the validation data. This threshold provides a better balance between precision and recall than the default 0.50 threshold.

The selected model is now retrained using all pre-test observations and evaluated once on the untouched final test period.

In [40]:
final_logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

final_logistic_model.fit(
    X_train,
    y_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](13,)","['order_value','freight_value','number_of_items',...,'purchase_month', 'purchase_weekday','promised_delivery_days']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,13
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``

In [41]:
final_test_probabilities = (
    final_logistic_model
    .predict_proba(X_test)[:, 1]
)

In [42]:
final_threshold = 0.65

final_test_predictions = (
    final_test_probabilities >= final_threshold
).astype(int)

In [43]:
final_accuracy = accuracy_score(
    y_test,
    final_test_predictions
)

final_precision = precision_score(
    y_test,
    final_test_predictions,
    zero_division=0
)

final_recall = recall_score(
    y_test,
    final_test_predictions,
    zero_division=0
)

final_f1 = f1_score(
    y_test,
    final_test_predictions,
    zero_division=0
)

final_roc_auc = roc_auc_score(
    y_test,
    final_test_probabilities
)

final_pr_auc = average_precision_score(
    y_test,
    final_test_probabilities
)

In [44]:
final_cm = confusion_matrix(
    y_test,
    final_test_predictions
)

tn, fp, fn, tp = final_cm.ravel()

In [45]:
print("=" * 60)
print("FINAL LOGISTIC REGRESSION - TEST SET")
print("=" * 60)

print(f"Threshold: {final_threshold:.2f}")
print(f"Test Late Rate: {y_test.mean() * 100:.2f}%")

print()
print("MODEL METRICS")
print("-" * 30)

print(f"Accuracy:  {final_accuracy:.3f}")
print(f"Precision: {final_precision:.3f}")
print(f"Recall:    {final_recall:.3f}")
print(f"F1 Score:  {final_f1:.3f}")
print(f"ROC-AUC:   {final_roc_auc:.3f}")
print(f"PR-AUC:    {final_pr_auc:.3f}")

print()
print("CONFUSION MATRIX")
print("-" * 30)

print(f"True Negatives:  {tn:,}")
print(f"False Positives: {fp:,}")
print(f"False Negatives: {fn:,}")
print(f"True Positives:  {tp:,}")

print()
print("FLAGGING BEHAVIOUR")
print("-" * 30)

flagged_orders = final_test_predictions.sum()

print(f"Orders in test set: {len(y_test):,}")
print(f"Orders flagged high-risk: {flagged_orders:,}")
print(
    f"Percentage flagged: "
    f"{flagged_orders / len(y_test) * 100:.2f}%"
)

FINAL LOGISTIC REGRESSION - TEST SET
Threshold: 0.65
Test Late Rate: 5.29%

MODEL METRICS
------------------------------
Accuracy:  0.803
Precision: 0.100
Recall:    0.340
F1 Score:  0.155
ROC-AUC:   0.717
PR-AUC:    0.115

CONFUSION MATRIX
------------------------------
True Negatives:  15,155
False Positives: 3,118
False Negatives: 674
True Positives:  347

FLAGGING BEHAVIOUR
------------------------------
Orders in test set: 19,294
Orders flagged high-risk: 3,465
Percentage flagged: 17.96%


## Model Interpretation

The final Logistic Regression model achieved a ROC-AUC of 0.717 on the chronologically held-out test period.

While classification performance was affected by a reduction in the prevalence of late deliveries over time, the model retained a moderate ability to rank higher-risk orders above lower-risk orders.

The model coefficients are examined below to identify characteristics associated with increased or reduced predicted delivery risk.

In [46]:
# Get feature names created by preprocessing

feature_names = (
    final_logistic_model
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

# Get Logistic Regression coefficients

coefficients = (
    final_logistic_model
    .named_steps["classifier"]
    .coef_[0]
)

coefficient_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})

# Clean up feature names to make them easier to read

coefficient_df["Feature"] = (
    coefficient_df["Feature"]
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
)

# Largest positive coefficients
# Positive = associated with HIGHER predicted late-delivery risk

top_positive = (
    coefficient_df
    .sort_values("Coefficient", ascending=False)
    .head(15)
)

# Largest negative coefficients
# Negative = associated with LOWER predicted late-delivery risk

top_negative = (
    coefficient_df
    .sort_values("Coefficient")
    .head(15)
)

print("=" * 70)
print("TOP FEATURES ASSOCIATED WITH HIGHER LATE-DELIVERY RISK")
print("=" * 70)

print(
    top_positive.to_string(
        index=False
    )
)

print("\n" + "=" * 70)
print("TOP FEATURES ASSOCIATED WITH LOWER LATE-DELIVERY RISK")
print("=" * 70)

print(
    top_negative.to_string(
        index=False
    )
)

TOP FEATURES ASSOCIATED WITH HIGHER LATE-DELIVERY RISK
                                                   Feature  Coefficient
                                   primary_seller_state_MA     2.000446
                                         customer_state_RR     1.724980
                   primary_product_category_home_comfort_2     1.182244
                                         customer_state_AL     1.179768
                                   primary_seller_state_AM     0.932990
                                         customer_state_PA     0.753883
                            primary_product_category_audio     0.645282
                                         customer_state_MA     0.626708
                                   primary_seller_state_PA     0.614564
               primary_product_category_christmas_supplies     0.588450
                                         customer_state_CE     0.564491
                                         customer_state_SE     0.561593
         

In [47]:
# RISK SEGMENT PERFORMANCE

risk_results = pd.DataFrame({
    "actual_late": y_test.values,
    "predicted_risk": final_test_probabilities
})

risk_results["risk_decile"] = pd.qcut(
    risk_results["predicted_risk"],
    q=10,
    labels=[
        "1 - Lowest",
        "2",
        "3",
        "4",
        "5",
        "6",
        "7",
        "8",
        "9",
        "10 - Highest"
    ]
)

risk_decile_summary = (
    risk_results
    .groupby(
        "risk_decile",
        observed=True
    )
    .agg(
        orders=("actual_late", "count"),
        actual_late_rate=("actual_late", "mean"),
        average_predicted_risk=("predicted_risk", "mean")
    )
    .reset_index()
)

risk_decile_summary["actual_late_rate"] *= 100
risk_decile_summary["average_predicted_risk"] *= 100

overall_test_late_rate = (
    risk_results["actual_late"].mean() * 100
)

highest_decile_late_rate = (
    risk_decile_summary
    .iloc[-1]["actual_late_rate"]
)

risk_lift = (
    highest_decile_late_rate /
    overall_test_late_rate
)

print("=" * 70)
print("RISK DECILE PERFORMANCE")
print("=" * 70)

print(
    risk_decile_summary.to_string(
        index=False
    )
)

print()
print(f"Overall Test Late Rate: {overall_test_late_rate:.2f}%")
print(
    f"Highest-Risk Decile Late Rate: "
    f"{highest_decile_late_rate:.2f}%"
)
print(
    f"Highest-Risk Decile Lift: "
    f"{risk_lift:.2f}x"
)

RISK DECILE PERFORMANCE
 risk_decile  orders  actual_late_rate  average_predicted_risk
  1 - Lowest    1930          0.310881               15.055853
           2    1929          1.088647               27.195387
           3    1929          2.073613               35.239607
           4    1930          2.435233               41.284963
           5    1929          2.954899               46.368899
           6    1929          5.391395               51.229205
           7    1930          7.772021               55.806981
           8    1929         10.627268               60.700234
           9    1929          9.745982               67.609644
10 - Highest    1930         10.518135               79.767524

Overall Test Late Rate: 5.29%
Highest-Risk Decile Late Rate: 10.52%
Highest-Risk Decile Lift: 1.99x


## Model Conclusions

Logistic Regression was selected over Random Forest because it achieved stronger validation performance while remaining more interpretable.

On the chronologically held-out test period, the final Logistic Regression model achieved a **ROC-AUC of 0.717**, indicating a moderate ability to rank higher-risk deliveries above lower-risk deliveries.

The prevalence of late delivery fell from **8.82% in the training period to 5.29% in the test period**, highlighting temporal change in delivery performance and reinforcing the use of chronological validation.

Using the validation-selected classification threshold of 0.65, the model achieved:

- **Precision:** 10.0%
- **Recall:** 34.0%
- **F1 Score:** 0.155
- **ROC-AUC:** 0.717

Although classification performance was limited by the low and changing prevalence of late deliveries, the model remained useful for risk prioritisation. Orders in the **highest predicted-risk decile had a 10.52% late-delivery rate compared with 5.29% overall**, representing approximately **1.99× lift**.

The model therefore appears more suitable as a tool for prioritising higher-risk orders for proactive review than as a definitive late-delivery classifier.

### Model Interpretation

Model coefficients suggest that geography is an important component of predicted delivery risk, with several customer and seller states appearing among the strongest positive and negative associations.

Longer promised delivery windows were associated with lower predicted late-delivery risk, consistent with the exploratory analysis showing lower observed late-delivery rates for orders with longer delivery windows.

Product category also contributed to predicted risk, although individual category and geographic coefficients should be interpreted cautiously because category frequencies vary substantially.

These relationships represent predictive associations rather than evidence of causation.

In [48]:
# CUMULATIVE HIGH-RISK CAPTURE

risk_ranked = risk_results.sort_values(
    "predicted_risk",
    ascending=False
).reset_index(drop=True)

total_late_orders = risk_ranked["actual_late"].sum()

for percentage in [10, 20, 30]:
    n_orders = int(
        len(risk_ranked) * (percentage / 100)
    )

    subset = risk_ranked.iloc[:n_orders]

    late_orders_captured = subset["actual_late"].sum()

    capture_rate = (
        late_orders_captured /
        total_late_orders *
        100
    )

    subset_late_rate = (
        subset["actual_late"].mean() *
        100
    )

    print(
        f"Top {percentage}% risk group:"
    )
    print(
        f"  Late orders captured: "
        f"{capture_rate:.2f}%"
    )
    print(
        f"  Actual late rate: "
        f"{subset_late_rate:.2f}%"
    )
    print()

Top 10% risk group:
  Late orders captured: 19.88%
  Actual late rate: 10.52%

Top 20% risk group:
  Late orders captured: 38.30%
  Actual late rate: 10.13%

Top 30% risk group:
  Late orders captured: 58.37%
  Actual late rate: 10.30%



In [49]:
import joblib

In [50]:
joblib.dump(
    final_logistic_model,
    "../models/late_delivery_model.joblib"
)

['../models/late_delivery_model.joblib']

In [51]:
joblib.dump(
    final_threshold,
    "../models/classification_threshold.joblib"
)

['../models/classification_threshold.joblib']